In [ ]:
%sql
-- SQL script for validating delivery_dt format and datatype in f_order table

/* 
   Implementation Logic:
   Validate the delivery_dt column in the f_order table to ensure
   it is 100% Decimal (38,0) and in yyyymmdd format.
*/

-- Create a temporary view to identify invalid delivery_dt entries
CREATE OR REPLACE TEMPORARY VIEW InvalidDeliveryDt AS
SELECT 
  order_nbr,
  delivery_dt
FROM purgo_playground.purgo_playground.f_order
WHERE NOT (
  delivery_dt IS NOT NULL AND 
  delivery_dt BETWEEN 20000101 AND 20301231 AND 
  CAST(delivery_dt AS STRING) RLIKE "^[0-9]{8}$"
);

-- Retrieve the count of invalid delivery_dt entries
SELECT 
  COUNT(*) AS invalid_count
FROM InvalidDeliveryDt;

-- Utilize a CTE to validate delivery_dt entries
WITH ValidationCheck AS (
  SELECT COUNT(*) AS invalid_count FROM InvalidDeliveryDt
)
SELECT CASE
  WHEN invalid_count = 0 THEN "Validation Passed: All delivery_dt entries are valid."
  ELSE "Error: There are invalid delivery_dt entries in the f_order table."
END AS validation_result
FROM ValidationCheck;

-- Clean up the temporary view
DROP VIEW IF EXISTS InvalidDeliveryDt;

-- Implement error handling for Delta Lake MERGE operation
MERGE INTO purgo_playground.purgo_playground.f_order AS target
USING (
  SELECT order_nbr, order_type, delivery_dt 
  FROM purgo_playground.purgo_playground.f_order
  WHERE delivery_dt = 99999999  -- Example of erroneous data
) AS source
ON target.order_nbr = source.order_nbr
WHEN MATCHED THEN 
  UPDATE SET delivery_dt = NULL  -- Replace with NULL or valid value
WHEN NOT MATCHED THEN 
  INSERT (order_nbr, order_type, delivery_dt) 
  VALUES (source.order_nbr, source.order_type, source.delivery_dt);

-- Validate results of the Delta MERGE operation
WITH DeltaValidation AS (
  SELECT COUNT(*) AS null_count 
  FROM purgo_playground.purgo_playground.f_order 
  WHERE delivery_dt IS NULL
)
SELECT CASE
  WHEN null_count > 0 THEN "Delta MERGE operation updated delivery_dt as expected."
  ELSE "Error: Delta MERGE operation did not update delivery_dt as expected."
END AS delta_operation_result
FROM DeltaValidation;
